In [ ]:
import os
import shutil
import numpy as np
import re
from collections import defaultdict
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem, Draw
from rdkit.Chem.Draw import IPythonConsole, rdMolDraw2D
#TODO I can't test the code if the data to run it is not in the repo.
#TODO Add a lot more comments about the "big brush stroke" logic like
# Read in the data
# Filter the structures based on the criteria
# Save the filtered results

### Get SMILES from log file

In [ ]:
#TODO This directory should be included as an example for people to run, and it should not be an absolute Path, but relative
#TODO Something like Path('./data/alkcl_props_sec/'), also switch to pathlib Path objects
#TODO The naming of the folders is inappropriate (run_it_back/MYWORK/)
sdf_directory = '/Users/theresewild/Sigman Group Dropbox/Therese Wild/haruka_echem_modeling/run_it_back/MYWORK/DFT_files/alkcl_props_sec'
log_names = []
with open('/Users/theresewild/Sigman Group Dropbox/Therese Wild/haruka_echem_modeling/run_it_back/MYWORK/DFT_files/alkcl_props_sec/log_ids.txt', 'r') as file:
    for line in file:
        com = line.strip()
        log_names.append(com)

rows = []

for log in log_names:
    base_name = log.split('.')[0]
    sdf_name = base_name + '.sdf'
    sdf_path = os.path.join(sdf_directory, sdf_name)
    suppl = Chem.SDMolSupplier(sdf_path, removeHs=False)
    original_mol = next((m for m in suppl if m is not None), None)

    if original_mol is None:
        print (f"sdf error for {sdf_name}")
        continue

    #TODO It's unclear what these lines of code are doing. If they're commented out, they should likely be removed
    # mol_fix_bond_order = enforce_substructure_bonds(
    #     sdf_name, original_mol, ['[N]1CC[N][Ni]1', '[N]1C=C[N][Ni]1', 'N1=CC=N[Ni]1']
    # )
    # mol_fix_pyridines = fix_pyridines(
    #     sdf_name, mol_fix_bond_order, ['C1=CC=CC[N]1', 'N1=C=CC=CC1', 'C1=CC=CC=N1']
    # )
    # no_metal_mol = remove_ni_and_neighbors(
    #     mol_fix_bond_order, remove_neighbors=True, neighbor_elements=["H"]
    # )
    smiles = Chem.MolToSmiles(original_mol)
    rows.append({"sdf_name": sdf_name, "log_smiles": smiles})

smiles_in_log = pd.DataFrame(rows)
smiles_in_log['id'] = smiles_in_log['sdf_name'].str.split('_').str[0]
files_list = list(smiles_in_log['id'])

In [ ]:
#TODO Is this for cannonical smiles? If so, the call MolToSmiles should have cannonical=True
def get_cannon(smiles):
    try:
        mol = Chem.MolFromSmiles(smiles)
        smi = Chem.MolToSmiles(mol)
    except:
        print (smiles)
    return smi

In [115]:
smiles_in_log['log_smiles'] = smiles_in_log['log_smiles'].apply(get_cannon)
smiles_in_log.drop_duplicates(subset='id', inplace=True)
smiles_in_log

,sdf_name,log_smiles,id
0,alkylCl04_1.sdf,ClCCCCOc1ccccc1,alkylCl04
5,alkylCl04.sdf,ClCCCCOc1ccccc1,alkylCl04.sdf
6,alkylCl17_conf-1.sdf,ClC1CCCCC1,alkylCl17
8,alkylCl18_conf-1.sdf,CC(C)(C)OC(=O)N1CCC(Cl)CC1,alkylCl18
12,alkylCl19_conf-1.sdf,CC(C)(C)OC(=O)N1CC[C@@H](Cl)C1,alkylCl19
15,alkylCl20_conf-1.sdf,Cl[C@H]1CCOC1,alkylCl20
17,alkylCl21_conf-1.sdf,CC(C)(C)OC(=O)N1CC(Cl)C1,alkylCl21


In [ ]:
smiles_in_log.to_excel('/Users/theresewild/Sigman Group Dropbox/Therese Wild/haruka_echem_modeling/run_it_back/MYWORK/DFT_files/alkcl_props_sec/smiles_in_log.xlsx', index=False)
#TODO Fix paths to be relative to the present directory

In [ ]:
#TODO Fix paths
props = pd.read_excel('/Users/theresewild/Sigman Group Dropbox/Therese Wild/haruka_echem_modeling/run_it_back/MYWORK/correction_factor/aryl/Summary_Properties_for_arbr_training_set.xlsx')
smiles_in_log = pd.read_excel('/Users/theresewild/Sigman Group Dropbox/Therese Wild/haruka_echem_modeling/run_it_back/MYWORK/DFT_files/arbr_props/arbr_log_smiles_other_ID.xlsx')
props.rename(columns={'Compound_Name': 'id'}, inplace=True)
smiles_in_log.drop(columns=['sdf_name'], inplace=True)
# smiles_in_log.rename(columns={'log_smiles': 'smiles'}, inplace=True)
print(smiles_in_log.shape)
print(props.shape)
merged = pd.merge(props, smiles_in_log, on='id')
display (merged)

(39, 2)
(39, 232)


,id,HOMO_Boltz,HOMO_Boltz_stdev,HOMO_min,HOMO_max,HOMO_range,HOMO_low_E,HOMO_V_bur_min,LUMO_Boltz,LUMO_Boltz_stdev,...,NBO_LP_occupancy_Br_low_E,NBO_LP_occupancy_Br_V_bur_min,NBO_LP_energy_Br_Boltz,NBO_LP_energy_Br_Boltz_stdev,NBO_LP_energy_Br_min,NBO_LP_energy_Br_max,NBO_LP_energy_Br_range,NBO_LP_energy_Br_low_E,NBO_LP_energy_Br_V_bur_min,smiles
0,arbr11119,-0.328810,0.000000,-0.32881,-0.32881,0.00000,-0.32881,-0.32881,-0.041290,0.000000,...,1.92854,1.92854,-0.354960,0.000000,-0.35496,-0.35496,0.00000,-0.35496,-0.35496,FC(F)(F)c1ccnc(Br)c1
1,arbr1409,-0.317570,0.000000,-0.31757,-0.31757,0.00000,-0.31757,-0.31757,-0.029510,0.000000,...,1.93794,1.93794,-0.356110,0.000000,-0.35611,-0.35611,0.00000,-0.35611,-0.35611,NS(=O)(=O)c1cccc(Br)c1
2,arbr1488,-0.282490,0.000000,-0.28249,-0.28249,0.00000,-0.28249,-0.28249,-0.018640,0.000000,...,1.94409,1.94409,-0.356800,0.000000,-0.35680,-0.35680,0.00000,-0.35680,-0.35680,Brc1cccc2cn[nH]c12
3,arbr1511,-0.286830,0.000000,-0.28683,-0.28683,0.00000,-0.28683,-0.28683,-0.041990,0.000000,...,1.93951,1.93951,-0.353110,0.000000,-0.35311,-0.35311,0.00000,-0.35311,-0.35311,Brc1cccc2cnccc12
4,arbr1547,-0.300660,0.000000,-0.30066,-0.30066,0.00000,-0.30066,-0.30066,0.005960,0.000000,...,1.94199,1.94199,-0.344280,0.000000,-0.34428,-0.34428,0.00000,-0.34428,-0.34428,Brc1ccccc1
5,arbr1648,-0.313210,0.000000,-0.31321,-0.31321,0.00000,-0.31321,-0.31321,-0.014710,0.000000,...,1.94037,1.94037,-0.354070,0.000000,-0.35407,-0.35407,0.00000,-0.35407,-0.35407,Brc1cccnc1
6,arbr1696,-0.293200,0.000000,-0.29320,-0.29320,0.00000,-0.29320,-0.29320,-0.043080,0.000000,...,1.93066,1.93066,-0.355500,0.000000,-0.35550,-0.35550,0.00000,-0.35550,-0.35550,Brc1ccnc2ccccc12
7,arbr186,-0.298900,0.000000,-0.29890,-0.29890,0.00000,-0.29890,-0.29890,0.033040,0.000000,...,1.95069,1.95069,-0.348880,0.000000,-0.34888,-0.34888,0.00000,-0.34888,-0.34888,Brc1cn[nH]c1
8,arbr1883,-0.266060,0.000000,-0.26606,-0.26606,0.00000,-0.26606,-0.26606,-0.018480,0.000000,...,1.94791,1.94791,-0.363160,0.000000,-0.36316,-0.36316,0.00000,-0.36316,-0.36316,Brc1cnc2ccccn12
9,arbr1959,-0.329690,0.000000,-0.32969,-0.32969,0.00000,-0.32969,-0.32969,-0.030590,0.000000,...,1.93837,1.93837,-0.364730,0.000000,-0.36473,-0.36473,0.00000,-0.36473,-0.36473,Brc1cncnc1


In [ ]:
#TODO Fix paths
#TODO Perhaps some markdown between these cells to give the user some idea of what is going on
merged.to_excel('/Users/theresewild/Sigman Group Dropbox/Therese Wild/haruka_echem_modeling/run_it_back/MYWORK/correction_factor/aryl/Summary_Properties_for_arbr_training_set_w_smiles.xlsx')

In [ ]:

#TODO duplicate function definition
def get_cannon(smiles):
    mol = Chem.MolFromSmiles(smiles)
    smi = Chem.MolToSmiles(mol)

    return smi

In [16]:
other_ids = pd.read_excel('/Users/theresewild/Sigman Group Dropbox/Therese Wild/haruka_echem_modeling/aryl_rates_full_data_w_lig_smiles_corrected_ligand_ids.xlsx')
other_ids = other_ids.filter(items=['ArID', 'aryl_canonical'])
other_ids.drop_duplicates(subset='ArID', inplace=True)
other_ids.rename(columns={'aryl_canonical': 'smiles'}, inplace=True)
display (other_ids)

,ArID,smiles
0,ArX-001,Clc1cccnc1
1,ArX-002,COC(=O)c1ccncc1Cl
2,ArX-004,CC(=O)c1ccccc1Cl
3,ArX-005,Clc1ccccc1
4,ArX-008,CCOc1ncccc1Cl
5,ArX-009,CCOC(=O)c1ccc(Cl)s1
6,ArX-011,Clc1cncnc1
7,ArX-012,Fc1ccc(Cl)cc1
8,ArX-013,CCOC(=O)c1cc(Cl)ccn1
9,ArX-014,Clc1cccc2cn[nH]c12


In [18]:
merged['smiles'] = merged['smiles'].apply(get_cannon)
other_ids['smiles'] = other_ids['smiles'].apply(get_cannon)

print (merged.shape)
print (other_ids.shape)

final = pd.merge (other_ids, merged, on='smiles')
display (final)

(39, 233)
(44, 2)


,ArID,smiles,id,HOMO_Boltz,HOMO_Boltz_stdev,HOMO_min,HOMO_max,HOMO_range,HOMO_low_E,HOMO_V_bur_min,...,NBO_LP_occupancy_Br_range,NBO_LP_occupancy_Br_low_E,NBO_LP_occupancy_Br_V_bur_min,NBO_LP_energy_Br_Boltz,NBO_LP_energy_Br_Boltz_stdev,NBO_LP_energy_Br_min,NBO_LP_energy_Br_max,NBO_LP_energy_Br_range,NBO_LP_energy_Br_low_E,NBO_LP_energy_Br_V_bur_min
0,ArX-042,FC(F)(F)c1ccnc(Br)c1,arbr11119,-0.328810,0.000000,-0.32881,-0.32881,0.00000,-0.32881,-0.32881,...,0.00000,1.92854,1.92854,-0.354960,0.000000,-0.35496,-0.35496,0.00000,-0.35496,-0.35496
1,ArX-046,Brc1cnc2ccccn12,arbr1883,-0.266060,0.000000,-0.26606,-0.26606,0.00000,-0.26606,-0.26606,...,0.00000,1.94791,1.94791,-0.363160,0.000000,-0.36316,-0.36316,0.00000,-0.36316,-0.36316
2,ArX-049,N#Cc1cc(Br)ccn1,arbr312,-0.348330,0.000000,-0.34833,-0.34833,0.00000,-0.34833,-0.34833,...,0.00000,1.92513,1.92513,-0.372520,0.000000,-0.37252,-0.37252,0.00000,-0.37252,-0.37252
3,ArX-040,Cc1ccc(Br)cc1,arbr4928,-0.291070,0.000000,-0.29107,-0.29107,0.00000,-0.29107,-0.29107,...,0.00000,1.94418,1.94418,-0.341770,0.000000,-0.34177,-0.34177,0.00000,-0.34177,-0.34177
4,ArX-041,COc1ncc(Br)c(OC)n1,arbr4696,-0.294117,0.000865,-0.29436,-0.29258,0.00178,-0.29436,-0.29258,...,0.00067,1.94638,1.94705,-0.350595,0.000053,-0.35069,-0.35058,0.00011,-0.35058,-0.35069
5,ArX-043,Brc1cccc2cnccc12,arbr1511,-0.286830,0.000000,-0.28683,-0.28683,0.00000,-0.28683,-0.28683,...,0.00000,1.93951,1.93951,-0.353110,0.000000,-0.35311,-0.35311,0.00000,-0.35311,-0.35311
6,ArX-044,CCOC(=O)c1ccc(Br)cc1,arbr3119,-0.309169,0.000064,-0.30924,-0.30914,0.00010,-0.30914,-0.30924,...,0.00012,1.93429,1.93417,-0.351077,0.000006,-0.35108,-0.35107,0.00001,-0.35108,-0.35107
7,ArX-045,COc1ccc(Br)cc1,arbr4505,-0.277670,0.000000,-0.27767,-0.27767,0.00000,-0.27767,-0.27767,...,0.00000,1.94810,1.94810,-0.341110,0.000000,-0.34111,-0.34111,0.00000,-0.34111,-0.34111
8,ArX-047,Cc1ccc(C)c(Br)c1,arbr4907,-0.290050,0.000000,-0.29005,-0.29005,0.00000,-0.29005,-0.29005,...,0.00000,1.94150,1.94150,-0.341350,0.000000,-0.34135,-0.34135,0.00000,-0.34135,-0.34135
9,ArX-048,FC(F)(F)c1ccc(Br)cc1,arbr5469,-0.320010,0.000000,-0.32001,-0.32001,0.00000,-0.32001,-0.32001,...,0.00000,1.93500,1.93500,-0.357310,0.000000,-0.35731,-0.35731,0.00000,-0.35731,-0.35731


In [11]:
smiles_in_log.to_excel('/Users/theresewild/Sigman Group Dropbox/Therese Wild/haruka_echem_modeling/run_it_back/MYWORK/DFT_files/arbr_props/arbr_log_smiles_other_ID.xlsx', index=False)

In [60]:
smiles = smiles_in_log
smiles.drop(columns=['sdf_name'], inplace=True)
smiles.rename(columns={'log_smiles':'smiles'}, inplace=True)
props = pd.read_excel('/Users/theresewild/Sigman Group Dropbox/Therese Wild/haruka_echem_modeling/run_it_back/MYWORK/Summary_Properties_for_alkylCl_training_set.xlsx')
props.rename(columns={'Compound_Name':'id'}, inplace=True)
display (props)
display (smiles)

,id,HOMO_Boltz,HOMO_Boltz_stdev,HOMO_min,HOMO_max,HOMO_range,HOMO_low_E,LUMO_Boltz,LUMO_Boltz_stdev,LUMO_min,...,Hirsh_CM5_charge_X_min,Hirsh_CM5_charge_X_max,Hirsh_CM5_charge_X_range,Hirsh_CM5_charge_X_low_E,Hirsh_atom_dipole_X_Boltz,Hirsh_atom_dipole_X_Boltz_stdev,Hirsh_atom_dipole_X_min,Hirsh_atom_dipole_X_max,Hirsh_atom_dipole_X_range,Hirsh_atom_dipole_X_low_E
0,alkylCl01,-0.372003,0.000300,-0.37378,-0.37013,0.00365,-0.37200,0.098594,0.003145,0.09438,...,-0.127493,-0.112912,0.014581,-0.121814,0.155731,0.000456,0.151080,0.155862,0.004782,0.155736
1,alkylCl02,-0.346276,0.001343,-0.35026,-0.34437,0.00589,-0.34591,0.059258,0.001527,0.05494,...,-0.120250,-0.091826,0.028424,-0.109651,0.126890,0.003693,0.112251,0.144918,0.032667,0.127085
2,alkylCl03,-0.305331,0.001543,-0.30659,-0.30115,0.00544,-0.30640,0.050856,0.001740,0.04874,...,-0.109642,-0.097277,0.012365,-0.104688,0.128162,0.001813,0.123206,0.129571,0.006366,0.126394
3,alkylCl04,-0.301991,0.002190,-0.30353,-0.29439,0.00914,-0.30192,0.053298,0.001809,0.05155,...,-0.123902,-0.099228,0.024674,-0.121589,0.147891,0.004380,0.133270,0.152952,0.019682,0.147138
4,alkylCl05,-0.365147,0.003424,-0.36701,-0.35815,0.00886,-0.36701,0.073148,0.004763,0.07093,...,-0.123740,-0.103852,0.019888,-0.123740,0.147009,0.007884,0.132180,0.153555,0.021375,0.153555
5,alkylCl06,-0.373270,0.000000,-0.37327,-0.37327,0.00000,-0.37327,0.090740,0.000000,0.09074,...,-0.110808,-0.110808,0.000000,-0.110808,0.151836,0.000000,0.151836,0.151836,0.000000,0.151836
6,alkylCl07,-0.352517,0.003537,-0.35422,-0.34759,0.00663,-0.35422,0.099310,0.001493,0.09859,...,-0.104463,-0.099690,0.004773,-0.104455,0.125005,0.000418,0.124801,0.125587,0.000786,0.124801
7,alkylCl08,-0.339490,0.002296,-0.34011,-0.32838,0.01173,-0.34011,0.016195,0.002123,0.01547,...,-0.115713,-0.100491,0.015222,-0.115713,0.136076,0.004531,0.120291,0.141195,0.020904,0.141195
8,alkylCl09,-0.328963,0.003442,-0.33038,-0.32059,0.00979,-0.33035,0.047417,0.003552,0.04610,...,-0.118699,-0.105683,0.013016,-0.111337,0.138555,0.003001,0.136478,0.142633,0.006155,0.136566
9,alkylCl10,-0.364341,0.001860,-0.36527,-0.35815,0.00712,-0.36483,0.099710,0.006505,0.08830,...,-0.127341,-0.092916,0.034425,-0.125307,0.157032,0.007749,0.131079,0.159841,0.028762,0.159841


,smiles,id
0,CCCCCCCl,alkylCl01
5,COC(=O)[C@@H](CCl)NC(C)=O,alkylCl02
17,ClCCOc1ccccc1,alkylCl03
28,ClCCCCOc1ccccc1,alkylCl04
33,CCOC(=O)CCCCl,alkylCl05
38,CC(C)(C)CCl,alkylCl06
39,ClCC1OCCO1,alkylCl07
42,Cc1ccc(S(=O)(=O)NCCCl)cc1,alkylCl08
47,ClCCOCc1ccccc1,alkylCl09
52,OCCCCl,alkylCl10


In [61]:
props_smiles = pd.merge(smiles, props, on='id')
props_smiles

,smiles,id,HOMO_Boltz,HOMO_Boltz_stdev,HOMO_min,HOMO_max,HOMO_range,HOMO_low_E,LUMO_Boltz,LUMO_Boltz_stdev,...,Hirsh_CM5_charge_X_min,Hirsh_CM5_charge_X_max,Hirsh_CM5_charge_X_range,Hirsh_CM5_charge_X_low_E,Hirsh_atom_dipole_X_Boltz,Hirsh_atom_dipole_X_Boltz_stdev,Hirsh_atom_dipole_X_min,Hirsh_atom_dipole_X_max,Hirsh_atom_dipole_X_range,Hirsh_atom_dipole_X_low_E
0,CCCCCCCl,alkylCl01,-0.372003,0.000300,-0.37378,-0.37013,0.00365,-0.37200,0.098594,0.003145,...,-0.127493,-0.112912,0.014581,-0.121814,0.155731,0.000456,0.151080,0.155862,0.004782,0.155736
1,COC(=O)[C@@H](CCl)NC(C)=O,alkylCl02,-0.346276,0.001343,-0.35026,-0.34437,0.00589,-0.34591,0.059258,0.001527,...,-0.120250,-0.091826,0.028424,-0.109651,0.126890,0.003693,0.112251,0.144918,0.032667,0.127085
2,ClCCOc1ccccc1,alkylCl03,-0.305331,0.001543,-0.30659,-0.30115,0.00544,-0.30640,0.050856,0.001740,...,-0.109642,-0.097277,0.012365,-0.104688,0.128162,0.001813,0.123206,0.129571,0.006366,0.126394
3,ClCCCCOc1ccccc1,alkylCl04,-0.301991,0.002190,-0.30353,-0.29439,0.00914,-0.30192,0.053298,0.001809,...,-0.123902,-0.099228,0.024674,-0.121589,0.147891,0.004380,0.133270,0.152952,0.019682,0.147138
4,CCOC(=O)CCCCl,alkylCl05,-0.365147,0.003424,-0.36701,-0.35815,0.00886,-0.36701,0.073148,0.004763,...,-0.123740,-0.103852,0.019888,-0.123740,0.147009,0.007884,0.132180,0.153555,0.021375,0.153555
5,CC(C)(C)CCl,alkylCl06,-0.373270,0.000000,-0.37327,-0.37327,0.00000,-0.37327,0.090740,0.000000,...,-0.110808,-0.110808,0.000000,-0.110808,0.151836,0.000000,0.151836,0.151836,0.000000,0.151836
6,ClCC1OCCO1,alkylCl07,-0.352517,0.003537,-0.35422,-0.34759,0.00663,-0.35422,0.099310,0.001493,...,-0.104463,-0.099690,0.004773,-0.104455,0.125005,0.000418,0.124801,0.125587,0.000786,0.124801
7,Cc1ccc(S(=O)(=O)NCCCl)cc1,alkylCl08,-0.339490,0.002296,-0.34011,-0.32838,0.01173,-0.34011,0.016195,0.002123,...,-0.115713,-0.100491,0.015222,-0.115713,0.136076,0.004531,0.120291,0.141195,0.020904,0.141195
8,ClCCOCc1ccccc1,alkylCl09,-0.328963,0.003442,-0.33038,-0.32059,0.00979,-0.33035,0.047417,0.003552,...,-0.118699,-0.105683,0.013016,-0.111337,0.138555,0.003001,0.136478,0.142633,0.006155,0.136566
9,OCCCCl,alkylCl10,-0.364341,0.001860,-0.36527,-0.35815,0.00712,-0.36483,0.099710,0.006505,...,-0.127341,-0.092916,0.034425,-0.125307,0.157032,0.007749,0.131079,0.159841,0.028762,0.159841


In [62]:
props_smiles.to_excel('/Users/theresewild/Sigman Group Dropbox/Therese Wild/haruka_echem_modeling/run_it_back/MYWORK/Summary_Properties_for_alkylCl_training_set_w_smiles.xlsx')

### Distortion Analysis

In [ ]:
import os
import math
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import rdMolTransforms

#TODO Ask ChatGPT to write some doc strings for these so we have some record of what they're doing
def angle(p1, p2, p3):
    v1 = p1 - p2
    v2 = p3 - p2
    cosang = np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))
    return np.degrees(np.arccos(np.clip(cosang, -1.0, 1.0)))

def compute_tau4_from_coords(metal_idx, neighbors, coords):
    angles = []
    for i in range(len(neighbors)):
        for j in range(i + 1, len(neighbors)):
            a = angle(
                coords[neighbors[i]],
                coords[metal_idx],
                coords[neighbors[j]]
            )
            angles.append(a)

    angles.sort(reverse=True)
    alpha, beta = angles[:2]
    tau4 = (360.0 - (alpha + beta)) / 141.0
    return tau4, alpha, beta

def process_sdf(file_path):
    mol = Chem.MolFromMolFile(file_path, removeHs=False)
    if mol is None:
        return None

    conf = mol.GetConformer()
    coords = np.array([conf.GetAtomPosition(i) for i in range(mol.GetNumAtoms())])

    # crude metal detection
    metals = [
        a.GetIdx() for a in mol.GetAtoms()
        if a.GetSymbol() not in ["H", "C", "N", "O", "S", "P", "F", "Cl", "Br", "I"]
    ]

    if len(metals) != 1:
        return None

    m = metals[0]
    neighbors = [n.GetIdx() for n in mol.GetAtomWithIdx(m).GetNeighbors()]

    if len(neighbors) != 4:
        return {
            "file": os.path.basename(file_path),
            "coordination": len(neighbors),
            "tau4": None,
            "alpha": None,
            "beta": None
        }

    tau4, alpha, beta = compute_tau4_from_coords(m, neighbors, coords)
    return {
        "file": os.path.basename(file_path),
        "coordination": 4,
        "tau4": tau4,
        "alpha": alpha,
        "beta": beta
    }

input_dir = '/Users/theresewild/Sigman Group Dropbox/Therese Wild/NN_Library/nn_ligand_modeling_datasets/all_single_stereochem_series/final/additional_modeling_information/distortion_metric/Structures Final_jules'

results = []

for file in os.listdir(input_dir):
    path = os.path.join(input_dir, file)

    if file.lower().endswith(".sdf"):
        res = process_sdf(path)
    else:
        continue

    if res:
        results.append(res)

df = pd.DataFrame(results)
df


,file,coordination,tau4,alpha,beta
0,csII_l12_III-b_R_left.sdf,5,NaN,NaN,NaN
1,csII_l12_III-b_S_left.sdf,5,NaN,NaN,NaN
2,csII_l24_rc-I_S_right.sdf,4,0.164006,170.684459,166.190761
3,csII_l24_re-b_S_left.sdf,4,0.655217,163.140407,104.474047
4,csII_l24_re-b_R_left.sdf,4,0.648150,158.468412,110.142457
5,csII_l12_rc-I_R_left.sdf,4,0.223349,167.922244,160.585508
6,csII_l12_rc-I_S_left.sdf,4,0.202531,170.685966,160.757185
7,csII_l24_rc-I_R_right.sdf,5,NaN,NaN,NaN
8,csII_l24_re-b_R_right.sdf,4,0.624011,159.667341,112.347057
9,csII_l12_III-b_R_right.sdf,5,NaN,NaN,NaN


In [ ]:
#TODO Add doc strings to functions
#TODO Move functions to a separate file called "tau_utils.py" or something like that and import the funcitons instead of defining here.

def all_LML_angles(metal_idx, neighbors, coords):
    angles = []
    for i in range(len(neighbors)):
        for j in range(i + 1, len(neighbors)):
            a = angle(
                coords[neighbors[i]],
                coords[metal_idx],
                coords[neighbors[j]]
            )
            angles.append(a)
    return sorted(angles, reverse=True)

def compute_tau4(angles):
    alpha, beta = angles[0], angles[1]
    tau4 = (360.0 - (alpha + beta)) / 141.0
    return tau4, alpha, beta

def compute_tau5(angles):
    beta, alpha = angles[0], angles[1]
    tau5 = (beta - alpha) / 60.0
    return tau5, beta, alpha

def process_sdf(file_path):
    mol = Chem.MolFromMolFile(file_path, removeHs=False)
    if mol is None:
        return None

    conf = mol.GetConformer()
    coords = np.array([conf.GetAtomPosition(i) for i in range(mol.GetNumAtoms())])

    metals = [
        a.GetIdx() for a in mol.GetAtoms()
        if a.GetSymbol() not in ["H", "C", "N", "O", "S", "P", "F", "Cl", "Br", "I"]
    ]

    if len(metals) != 1:
        return None

    m = metals[0]
    neighbors = [n.GetIdx() for n in mol.GetAtomWithIdx(m).GetNeighbors()]
    cn = len(neighbors)

    angles = all_LML_angles(m, neighbors, coords)

    result = {
        "file": os.path.basename(file_path),
        "coordination": cn,
        "tau4": None,
        "tau5": None,
        "alpha": None,
        "beta": None
    }

    if cn == 4:
        tau4, alpha, beta = compute_tau4(angles)
        result.update({
            "tau4": tau4,
            "alpha": alpha,
            "beta": beta
        })

    elif cn == 5:
        tau5, beta, alpha = compute_tau5(angles)
        result.update({
            "tau5": tau5,
            "alpha": alpha,
            "beta": beta
        })

    return result

#TODO Fix Paths
input_dir ='/Users/theresewild/Sigman Group Dropbox/Therese Wild/NN_Library/nn_ligand_modeling_datasets/all_single_stereochem_series/final/additional_requested_information/distortion_metric/Structures_final_CSI'

results = []

for file in os.listdir(input_dir):
    path = os.path.join(input_dir, file)

    if file.lower().endswith(".sdf"):
        res = process_sdf(path)
    else:
        continue

    if res:
        results.append(res)

df = pd.DataFrame(results)
df

,file,coordination,tau4,tau5,alpha,beta
0,CSI_L22_III-a_S_left.sdf,5,NaN,0.412004,144.047958,168.768228
1,CSI_L22_III-a_R_left.sdf,5,NaN,0.233389,150.442513,164.445863
2,CSI_L12_RC-I_R_right.sdf,5,NaN,0.176654,159.643891,170.243120
3,CSI_L12_RC-I_S_left.sdf,5,NaN,0.177497,155.795205,166.444999
4,CSI_L12_RC-I_R_left.sdf,5,NaN,0.115783,159.214925,166.161881
5,CSI_L22_RE-b_S_left.sdf,5,NaN,0.201837,146.547091,158.657319
6,CSI_L22_RE-b_R_left.sdf,5,NaN,0.206906,146.123371,158.537708
7,CSI_L12_RE-a_S_right.sdf,4,0.514716,NaN,152.133699,135.291396
8,CSI_L12_RC-I_S_right.sdf,5,NaN,0.108066,162.248210,168.732160
9,CSI_L22_RC-I_R_right.sdf,5,NaN,0.032561,159.983830,161.937462


In [224]:
df.to_excel(input_dir + '/TS_ligs_tau_values.xlsx')

### Get Missing NMR Values

In [ ]:
def extract_shielding_values(filename, directory):
    """Return dict: atom_number -> (isotropic, anisotropic)"""
    results = {}
    filepath = os.path.join(directory, filename)
    with open(filepath, "r") as f:
        lines = f.readlines()

    start = end = None

    for i, line in enumerate(lines):
        if "SCF GIAO Magnetic shielding tensor (ppm):" in line:
            start = i
        elif "g value of the free electron" in line and start is not None:
            end = i
            break

    if start is None or end is None:
        return results

    section = lines[start:end]

    pattern = re.compile(
        r"^\s*(\d+)\s+\w+\s+Isotropic\s*=\s*([-\d.]+)\s+Anisotropy\s*=\s*([-\d.]+)"
    )

    for line in section:
        m = pattern.search(line)
        if m:
            results[int(m.group(1))] = (
                float(m.group(2)),
                float(m.group(3))
            )
    return results

#TODO Fix Paths
directory = '/Users/theresewild/Sigman Group Dropbox/Therese Wild/NN_Library/dft_library_all/fluoride_library_ligands/new_basis_set/logs/pynx_matches_old_when_corrections_finish/'
df = pd.read_excel('/Users/theresewild/Sigman Group Dropbox/Therese Wild/NN_Library/dft_library_all/fluoride_library_ligands/new_basis_set/logs/pynx_matches_old_when_corrections_finish/pynx_properties_raw.xlsx',
                 sheet_name='Sheet2')

atom_cols = ["C1", "C2", "N1", "N2"]
for col in atom_cols:
    df[f"NMR_shift_{col}"] = None
    df[f"aniso_NMR_shift_{col}"] = None
for idx, row in df.iterrows():
    log_base_name = row["log_name"]
    log_name = f"{log_base_name}.log"
    values = extract_shielding_values(log_name, directory)
    for col in atom_cols:
        atom = row[col]
        print (atom)
        print (values)
        if atom in values:
            iso, aniso = values[atom]
            print (iso, aniso)
            df.at[idx, f"NMR_shift_{col}"] = iso
            df.at[idx, f"aniso_NMR_shift_{col}"] = aniso
df.to_excel(directory + '/extra_nmr.xlsx', index=False)

8
{1: (73.1417, 2051.2527), 2: (208.9781, 7388.4216), 3: (554.6932, 6401.285), 4: (-7135.5517, 14628.2578), 5: (-19032.4618, 46274.306), 6: (847.2357, 2368.6065), 7: (1142.5769, 4036.0794), 8: (-1677.5113, 16921.0838), 9: (-4069.8364, 24476.7351), 10: (8397.3201, 90149.9239), 11: (-3463.4401, 15648.6793), 12: (-356.8293, 7540.3343), 13: (-507.4977, 3161.1237), 14: (70.6008, 1741.5484), 15: (169.7875, 1314.0307), 16: (1742.8271, 9904.2691), 17: (3090.7963, 8870.2621), 18: (1270.5048, 3319.8854), 19: (-1251.3336, 5234.7701), 20: (-7565.1665, 12227.6898), 21: (825.5268, 3498.2838), 22: (954.7185, 3095.0716), 23: (522.7131, 1464.1585), 24: (323.5373, 982.6599), 25: (276.1835, 1121.7407), 26: (784.0381, 2959.2375), 27: (3.2895, 1553.5718), 28: (-165.9146, 1445.4812), 29: (-265.3174, 1235.1495), 30: (-8103.3709, 16291.26), 31: (403.5073, 2316.5186), 32: (1208.8588, 3552.7307), 33: (-6439.6824, 11697.6734), 34: (464.3287, 5422.9842), 35: (-819.2795, 2570.8748), 36: (-759.9586, 3215.313), 37: 

In [ ]:
import re

def extract_shielding_values(filename, directory):
    results = {}

    filepath = os.path.join(directory, filename)
    with open(filepath, "r") as f:
        lines = f.readlines()

    start = end = None
    for i, line in enumerate(lines):
        if "SCF GIAO Magnetic shielding tensor (ppm):" in line:
            start = i
        elif "g value of the free electron" in line and start is not None:
            end = i
            break

    if start is None or end is None:
        return results

    section = lines[start:end]

    pattern = re.compile(
        r"^\s*(\d+)\s+\w+\s+Isotropic\s*=\s*([^\s]+)\s+Anisotropy\s*=\s*([^\s]+)"
    )

    for line in section:
        m = pattern.search(line)
        if not m:
            continue

        atom = int(m.group(1))
        iso_raw = m.group(2)
        aniso_raw = m.group(3)

        try:
            iso = float(iso_raw)
            aniso = float(aniso_raw)
            results[atom] = (iso, aniso)

        except ValueError:

            print(f"Unreadable shielding values in {filename}:")
            print(line.rstrip())

            results[atom] = (None, None)

    return results

directory = '/Users/theresewild/Sigman Group Dropbox/Therese Wild/NN_Library/dft_library_all/fluoride_library_ligands/fluoride_logs/biox'
df = pd.read_excel('/Users/theresewild/Sigman Group Dropbox/Therese Wild/NN_Library/dft_library_all/fluoride_library_ligands/fluoride_logs/biox/biox_properties_raw.xlsx',
                 sheet_name='Sheet2')

atom_cols = ["C1", "C2", "N1", "N2"]
for col in atom_cols:
    df[f"{col}_iso"] = None
    df[f"{col}_aniso"] = None
for idx, row in df.iterrows():
    log_base_name = row["log_name"]
    log_name = f"{log_base_name}.log"
    values = extract_shielding_values(log_name, directory)
    for col in atom_cols:
        atom = row[col]
        if atom in values:
            iso, aniso = values[atom]
            df.at[idx, f"{col}_iso"] = iso
            df.at[idx, f"{col}_aniso"] = aniso
df


Unreadable shielding values in Lig218_xyz_12-1.log:
      1  N    Isotropic =***********   Anisotropy =297308.6570
Unreadable shielding values in Lig218_xyz_12-1.log:
     19  O    Isotropic =***********   Anisotropy =395967.8152
Unreadable shielding values in Lig218_xyz_12-1.log:
     73  Ni   Isotropic =***********   Anisotropy =***********
Unreadable shielding values in Lig218_xyz_12-1.log:
     74  F    Isotropic =***********   Anisotropy =***********
Unreadable shielding values in Lig218_xyz_12-1.log:
     75  F    Isotropic =***********   Anisotropy =***********
Unreadable shielding values in Lig39_xyz_6.log:
      1  O    Isotropic =145937.3669   Anisotropy =***********
Unreadable shielding values in Lig39_xyz_6.log:
      2  C    Isotropic =205935.8063   Anisotropy =***********
Unreadable shielding values in Lig39_xyz_6.log:
      3  C    Isotropic =***********   Anisotropy =***********
Unreadable shielding values in Lig39_xyz_6.log:
      4  C    Isotropic =***********   Aniso

,log_name,F1,Ni,N2,C2,C1,N1,F2,E_spc (Hartree),ZPE(Hartree),...,NMR_shift_N2,aniso_NMR_shift_N2,C1_iso,C1_aniso,C2_iso,C2_aniso,N1_iso,N1_aniso,N2_iso,N2_aniso
0,Lig218_xyz_12-1,74,73,16,15,12,1,75,-2908.739769,0.651579,...,no data,no data,3028.3396,34550.6707,46965.9404,208604.7656,None,None,-18838.094,101542.6918
1,Lig39_xyz_6,52,51,8,9,6,7,53,-2969.055966,0.416219,...,no data,no data,None,None,None,None,None,None,None,None
2,Lig46_xyz_15-1,58,57,1,14,15,16,59,-2899.088680,0.476462,...,no data,no data,None,None,-99734.9923,193437.8897,None,None,None,None
3,Lig47_xyz_20-1,56,55,13,12,11,1,57,-2672.946843,0.485097,...,no data,no data,-1518.5044,435186.076,11064.7642,312676.9649,-53326.605,630876.4081,-90776.6705,847457.0867
4,Lig54_xyz_11-1,54,53,17,16,2,1,55,-3004.987186,0.421129,...,no data,no data,-25514.9503,88825.9169,16953.3173,76520.0905,None,None,90571.624,281900.4923


In [ ]:
#TODO Fix Paths
df.to_excel('/Users/theresewild/Sigman Group Dropbox/Therese Wild/NN_Library/dft_library_all/fluoride_library_ligands/fluoride_logs/biox/sample.xlsx')

### Free ligand Gen

In [ ]:
import os
import shutil

#TODO Fix Paths
#TODO Move functions elsewhere since we don't know precisely what they're doing (no doc strings)
work_dir ='/Users/theresewild/Sigman Group Dropbox/Therese Wild/NN_Library/dft_library_all/hydride_library_ligands/updated_basis_set/bpy/logs/problem_geometries'
if not os.path.isdir(work_dir):
    raise ValueError("Directory does not exist")

problem_dir = os.path.join(work_dir, "problem_geometries")
os.makedirs(problem_dir, exist_ok=True)
def get_outstreams(log_path):
    """gets the compressed stream information at the end of a Gaussian job"""
    streams = []
    starts, ends = [], []
    error = "failed or incomplete job"

    try:
        with open(log_path) as f:
            loglines = f.readlines()
    except FileNotFoundError:
        raise FileNotFoundError(f"Cannot open {log_path}")

    for i, line in enumerate(loglines):
        if "1\\1\\" in line:
            starts.append(i)
        if "@" in line:
            ends.append(i)
        if "Normal termination" in line:
            error = ""

    if len(starts) != len(ends) or len(starts) == 0:
        return streams, "failed or incomplete job"

    for i in range(len(starts)):
        tmp = ""
        for j in range(starts[i], ends[i] + 1):
            tmp += loglines[j][1:-1]
        streams.append(tmp.split("\\"))

    return streams, error

def get_geom(streams):
    """extracts the geometry from the compressed stream"""
    geom = []
    try:
        for item in streams[-1][16:]:
            if item == "":
                break
            parts = item.split(",")
            geom.append([
                parts[0],
                float(parts[-3]),
                float(parts[-2]),
                float(parts[-1])
            ])
        return geom
    except Exception as e:
        print("Geometry extraction error:", e)
        return None

files_uncurated = os.listdir(work_dir)

log_files = [
    f[:-4] for f in files_uncurated
    if f.lower().endswith(".log") and "SPE" not in f
]

print(f"Processing {len(log_files)} log files")

for cat_name in log_files:
    log_path = os.path.join(work_dir, cat_name + ".log")

    streams, errors = get_outstreams(log_path)
    geometry = get_geom(streams)

    if geometry is None:
        print(f"geometry missing from input file {cat_name}")
        continue

    geometry_ok = True
    if geometry[-3][0] == 'Ni':
        geometry.pop(-1)
        geometry.pop(-1)
        geometry.pop(-1)
    elif geometry[-1][0] == 'Ni':
        geometry.pop(-1)
        geometry.pop(-1)
        geometry.pop(-1)
    else:
        print(f"metal geometry located elsewhere for file {cat_name}")
        geometry_ok = False

    free_filename = cat_name + "_free.com"
    free_path = os.path.join(work_dir, free_filename)

    with open(free_path, 'w') as new_com:
        new_com.write('# Put Keywords Here, check Charge and Multiplicity.\n\n')
        new_com.write(' title \n\n')
        new_com.write('0 1 \n')
        for atom in geometry:
            new_com.write(
                f"{atom[0]}\t{atom[1]}\t{atom[2]}\t{atom[3]}\n"
            )
        new_com.write('\n')

    # === MOVE PROBLEM FILES ===
    if not geometry_ok:
        shutil.move(log_path, os.path.join(problem_dir, os.path.basename(log_path)))
        shutil.move(free_path, os.path.join(problem_dir, free_filename))

Processing 1 log files
Geometry extraction error: list index out of range
geometry missing from input file Lig1192_xyz-1


### Draw SDF in 2 columns

### Get Elements + XYZ coordinates from an XYZ File

In [ ]:
from morfeus import read_xyz

#TODO Fix Paths

directory = '/Users/theresewild/Sigman Group Dropbox/Therese Wild/merck_collab/dft_calculations/arbr_lib_recalculations/dft_calculations/logs/class/jules/group9'
molecule_ids = []
conf_ids = []
elements_per_mol = []
coordinates_per_mol = []
for filename in os.listdir(directory):
    if filename.endswith('.xyz'):
        mol_id = filename.split('_')[0]
        conf_id = filename.split('_')[1].split('.')[0]
        elements, coordinates = read_xyz(os.path.join(directory, filename))

        elements_per_mol.append(elements)
        coordinates_per_mol.append(coordinates)
        molecule_ids.append(mol_id)
        conf_ids.append(conf_id)

conformers_df = pd.DataFrame({
    'molecule_id': molecule_ids,
    'conf_id': conf_ids,
    'elements': elements_per_mol,
    'coordinates': coordinates_per_mol})

conformers_df['basis'] = '6-31G(d,p)'
conformers_df['program'] = 'Gaussian 16, Revision C.01'
conformers_df['method'] = 'B3LYP-D3(BJ)'
conformers_df['cluster'] = 'FALSE'

display (conformers_df)

,molecule_id,conf_id,elements,coordinates,basis,program,method,cluster
0,arbr11164,conf-10,"[C, C, C, C, O, C, O, N, C, C, N, C, C, C, C, ...","[[-5.95406, -0.87043, -0.15211], [-4.57664, -1...","6-31G(d,p)","Gaussian 16, Revision C.01",B3LYP-D3(BJ),FALSE
1,arbr10776,conf-10,"[O, S, O, C, C, C, C, N, C, Br, N, C, C, C, C,...","[[0.55881, 0.24089, 2.1913], [0.53454, 1.01459...","6-31G(d,p)","Gaussian 16, Revision C.01",B3LYP-D3(BJ),FALSE
2,arbr6948,clust-2,"[C, C, O, C, O, C, C, N, C, N, C, C, O, C, C, ...","[[5.42197, -0.37411, -0.50254], [4.71956, -0.6...","6-31G(d,p)","Gaussian 16, Revision C.01",B3LYP-D3(BJ),FALSE
3,arbr7825,clust-11,"[C, C, C, C, C, C, C, O, C, O, C, O, C, C, C, ...","[[-5.21371, 2.90599, 2.28081], [-5.44402, 2.19...","6-31G(d,p)","Gaussian 16, Revision C.01",B3LYP-D3(BJ),FALSE
4,arbr12070,conf-10,"[C, O, C, O, C, C, C, C, O, C, C, C, C, C, C, ...","[[-6.36323, -1.56545, 0.0945], [-4.92934, -1.5...","6-31G(d,p)","Gaussian 16, Revision C.01",B3LYP-D3(BJ),FALSE
...,...,...,...,...,...,...,...,...
1937,arbr10853,clust-9,"[C, C, C, C, C, O, C, N, C, C, C, C, C, Br, H,...","[[-3.65836, 0.63543, -0.20312], [-3.72893, -0....","6-31G(d,p)","Gaussian 16, Revision C.01",B3LYP-D3(BJ),FALSE
1938,arbr9823,conf-4,"[Br, C, C, C, O, C, C, C, C, C, C, C, C, N, N,...","[[-4.11794, -1.04167, -0.00066], [-2.64918, 0....","6-31G(d,p)","Gaussian 16, Revision C.01",B3LYP-D3(BJ),FALSE
1939,arbr10193,conf-5,"[C, C, O, C, O, C, C, C, C, C, Br, C, C, C, N,...","[[5.9037, -1.29707, 0.99978], [5.46709, -0.813...","6-31G(d,p)","Gaussian 16, Revision C.01",B3LYP-D3(BJ),FALSE
1940,arbr10371,1,"[O, S, O, O, N, Br, C, C, C, C, C, C, H, H, H,...","[[-2.49868, -1.2686, -0.93277], [-2.18261, -0....","6-31G(d,p)","Gaussian 16, Revision C.01",B3LYP-D3(BJ),FALSE


In [30]:
conformers_df.to_excel('/Users/theresewild/Sigman Group Dropbox/Therese Wild/merck_collab/dft_calculations/arbr_lib_recalculations/dft_calculations/logs/class/jules/conf_tableg9.xlsx', index=False)

### Filtering Ligands but Keeping them In DF

In [ ]:
#TODO These functions are define here but it's not clear what they are doing or will do since they're not called here
#TODO Add doc strings

def molecular_weight_filter(df, weight_limit=500.0):
    """
    Flags molecules with molecular weight above the specified limit.
    Modifies 'library_status' and 'status_notes' columns in-place.
    """
    removed = 0

    for i, row in df.iterrows():
        if row['library_status'] == 'removed':
            continue  # Skip already removed

        smi = row['SMILES']
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            df.at[i, 'library_status'] = 'removed'
            df.at[i, 'status_notes'] = 'invalid SMILES'
            removed += 1
            continue

        weight = Descriptors.ExactMolWt(mol)
        if weight >= weight_limit:
            df.at[i, 'library_status'] = 'removed'
            df.at[i, 'status_notes'] = f'molecular weight {weight:.2f} exceeds limit ({weight_limit})'
            removed += 1

    print(f"{removed} ligands over molecular weight limit were marked for removal.")
    return df



from rdkit import Chem

def append_reason(df, idx, new_reason, status_col='library_status', reason_col='status_notes'):
    current_reason = df.at[idx, reason_col]
    if pd.isna(current_reason) or current_reason == '':
        df.at[idx, reason_col] = new_reason
    else:
        df.at[idx, reason_col] += f"; {new_reason}"
    df.at[idx, status_col] = 'removed'




def forbidden_substructures_filter(df):
    # SMARTS patterns for forbidden substructures
    forbidden = [
        Chem.MolFromSmarts(p) for p in [
            'c1cc(C2=NCCO2)nc(C2=NCCO2)c1',  # pybox
            'c1ccc(-c2cccc(C3=NCCO3)n2)nc1',  # bybox
            'c1ccc(-c2cccc(-c3ccccn3)n2)nc1',  # byby
            'c1(c2c(c3ncccc3)nccc2)ncccc1',  # triby
            'c1cnc2c(c1)ccc1ccc(C3=NCCO3)nc12',  # phenBox
            'c1cnc2nc(C3=NCCO3)ccc2c1',  # weirdbpy
            'c1ccc(C2CCCCN2)nc1',  # fakebpy
            'OB(*)O',  # boronic acids
            'O=C(*)N*',  # amides
            '[#6][OH]',  # free alcohol
            '[#6][NH2]',  # free amines
            '[#6]I',  # iodides
            '*P(O)(O)=O',  # phosphates
            '*C(O)=O',  # carboxylic acids
            '*C(Cl)=O',  # acid chlorides
            '*C#C*',  # alkynes
            '*C#C',  # terminal alkynes
            '[#6][NH][NH2]',  # N amines
            'c1ccsc1',  # sulfur heterocycle
            '[#6][SH]',  # free thiol
            'CN1CCOCCOCCOCCOCCOCC1',  # chelates
            'O=S=O',  # sulfones
            '*S*',  # general sulfur
            'c1(c2nc(C3[*]CCS3)ccc2)ncccc1',  # snn tri
            '*1:*c(-c2cccc(-c3ccccn3)n2)ncc1',  # nnn tri
            'S=C=N[*]',  # SCN
            '[N]=C=O',  # NCO
            '[O-]',  # O-
            '*[Se]*',  # selenium
            'C=C=C',  # allene
        ]
    ]
    # RDKit quirks
    forbidden[20].GetAtomWithIdx(1).SetNoImplicit(True)
    forbidden[21].GetAtomWithIdx(1).SetNoImplicit(True)
    forbidden[22].GetAtomWithIdx(1).SetNoImplicit(True)

    removed = 0

    for i, row in df.iterrows():
        if row['library_status'] == 'removed':
            continue

        smi = row['SMILES']
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            append_reason(df, i, 'invalid SMILES')
            removed += 1
            continue

        for sub in forbidden:
            if mol.HasSubstructMatch(sub):
                append_reason(df, i, 'forbidden substructure')
                removed += 1
                break

    print(f"{removed} ligands had forbidden substructures and were marked for removal.")
    return df

def multiple_binding_sites_filter(df):
    forbidden_if_doubles = [
        Chem.MolFromSmarts(p) for p in [
            'c1(c2ncccc2)ccccn1',  # bpy
            'C1COC(C2=NCCO2)=N1',  # biox
            'C1COC(CC2=NCCO2)=N1',  # box
            'C1CNC(C2=NCCN2)=N1',  # bilm
            'c1ccc(C2=NCCO2)nc1',  # pyox
            'C1=CN=C(c2ccccn2)[N]1',  # pyNx
        ]
    ]

    removed = 0

    for i, row in df.iterrows():
        if row['library_status'] == 'removed':
            continue

        smi = row['SMILES']
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            append_reason(df, i, 'invalid SMILES')
            removed += 1
            continue

        # Phase 1: remove if it matches multiple *unique* substructures
        found_subs = set()
        for sub in forbidden_if_doubles:
            if mol.HasSubstructMatch(sub):
                found_subs.add(sub)
            if len(found_subs) >= 2:
                append_reason(df, i, 'contains multiple possible binding sites')
                removed += 1
                break
        if df.at[i, 'library_status'] == 'removed':
            continue  # already flagged

        # Phase 2: remove if it has multiple copies of the same substructure
        for sub in forbidden_if_doubles:
            matches = mol.GetSubstructMatches(sub)
            if len(matches) > 1:
                append_reason(df, i, 'multiple binding site repeats')
                removed += 1
                break

    print(f"{removed} ligands had multiple binding sites and were marked for removal.")
    return df

def missing_bind_site_filter(df):
    required = [
        Chem.MolFromSmarts(p) for p in [
            'c1(c2ncccc2)ccccn1',  # bpy
            'C1COC(C2=NCCO2)=N1',  # biox
            'C1COC(CC2=NCCO2)=N1',  # box
            'C1CNC(C2=NCCN2)=N1',  # bilm
            'c1cnc2c(c1)ccc1cccnc12',  # phen
            'c1ccc(C2=NCCO2)nc1',  # pyox
            'C1=CN=C(c2ccccn2)[N]1',  # pyNx
            'c1cnc2c(C3=NCCN3)cccc2c1'  # bnx
        ]
    ]

    removed = 0

    for i, row in df.iterrows():
        if row['library_status'] == 'removed':
            continue

        mol = Chem.MolFromSmiles(row['SMILES'])
        if mol is None:
            append_reason(df, i, 'invalid SMILES')
            removed += 1
            continue

        if not any(mol.HasSubstructMatch(p) for p in required):
            append_reason(df, i, 'missing required binding site')
            removed += 1

    print(f"{removed} ligands are missing a binding site and were marked for removal.")
    return df

def heterocycle_filter(df):
    dummy_het = [
        Chem.MolFromSmarts(p) for p in [
            '*n1nnn(*)c1=O', '*c1cn(*)nn1', '*c1ncn(*)n1', '*c1ncn(*)n1',
            '*n1cnc(=O)[nH]1', '*C1=NN=C(*)[*]1', '*n1cccn1', '*c1nc[nH]n1',
            '*c1nnn[nH]1', '*c1c[nH]cn1', 'c1cn[nH]c1', '*c1ncoc1*',
            'Cn1ccnn1', 'CC1=NN=C(C)O1', 'Cn1cnn(C)c1=O',
            'c1(c2nc(C3C[N]CO3)ccc2)nc(C4C[N]CO4)ccc1',
            'CCOc1nc(N2CCN(CC2)Cc3cnc(c4ncccc4)cc3)ncc1',
            '*NCc1cccc(-c2ccccn2)n1', 'C[#7]c1cccc(c2ncccc2)n1',
            'c1ccc(-c2ccc3c(n2)NCC3)nc1', '[*]Nc1nc(c2ncccc2)ccc1'
        ]
    ]

    removed = 0

    for i, row in df.iterrows():
        if row['library_status'] == 'removed':
            continue

        mol = Chem.MolFromSmiles(row['SMILES'])
        if mol is None:
            append_reason(df, i, 'invalid SMILES')
            removed += 1
            continue

        if any(mol.HasSubstructMatch(p) for p in dummy_het):
            append_reason(df, i, 'extra nitrogen heterocycles - possible multiple binding sites')
            removed += 1

    print(f"{removed} ligands contained problematic nitrogen heterocycles and were marked for removal.")
    return df

def isotopes_filter(df):
    removed = 0
    smiles_seen = set()

    for i, row in df.iterrows():
        if row['library_status'] == 'removed':
            continue

        mol = Chem.MolFromSmiles(row['SMILES'])
        if mol is None:
            append_reason(df, i, 'invalid SMILES')
            removed += 1
            continue

        for atom in mol.GetAtoms():
            if atom.GetIsotope():
                atom.SetIsotope(0)

        clean_smi = Chem.MolToSmiles(mol)

        if clean_smi in smiles_seen:
            append_reason(df, i, 'duplicate when considering isotopes')
            removed += 1
        else:
            smiles_seen.add(clean_smi)

    print(f"{removed} ligands were duplicates due to isotope encoding and were marked for removal.")
    return df

def is_metal(atom):
    n = atom.GetAtomicNum()
    return (n == 5) or (21 <= n <= 34) or (37 <= n <= 52) or (n >= 54)

def other_metals_filter(df):
    removed = 0

    for i, row in df.iterrows():
        if row['library_status'] == 'removed':
            continue

        mol = Chem.MolFromSmiles(row['SMILES'])
        if mol is None:
            append_reason(df, i, 'invalid SMILES')
            removed += 1
            continue

        if any(is_metal(atom) for atom in mol.GetAtoms()):
            append_reason(df, i, 'ligand contains metal outside bonding site')
            removed += 1

    print(f"{removed} ligands contained metals and were marked for removal.")
    return df




In [172]:
df = pd.read_excel('/Users/therese/Sigman Group Dropbox/Therese Wild/NN_Library/dft_library_all/nn_ligand_library_smiles_w_notes_class_stereochem_8.7.25.xlsx',
                   sheet_name='all')
df

,SMILES,id,library_status,status_notes,ligand_class,stereochem
0,C[C@H]1COC(C2=N[C@@H](C)CO2)=N1,Lig1,in library,NaN,biox,"S,S"
1,CC(C)[C@H]1COC(C2=N[C@@H](C(C)C)CO2)=N1,Lig2,in library,NaN,biox,"S,S"
2,CCCC(CCC)[C@H]1COC(C2=N[C@@H](C(CCC)CCC)CO2)=N1,Lig3,in library,NaN,biox,"S,S"
3,CC(C)C[C@H]1COC(C2=N[C@@H](CC(C)C)CO2)=N1,Lig4,in library,NaN,biox,"S,S"
4,c1ccc([C@H]2COC(C3=N[C@@H](c4ccccc4)CO3)=N2)cc1,Lig5,in library,NaN,biox,"S,S"
...,...,...,...,...,...,...
2046,CCCCC(CCCC)[C@H]1COC(C2=N[C@@H](C(CCCC)CCCC)CO...,Lig2353,in library,NaN,biox,"S,S"
2047,CC(C)CC(CC(C)C)[C@H]1COC(C2=N[C@@H](C(CC(C)C)C...,Lig2354,in library,NaN,biox,"S,S"
2048,CC[C@H](C)[C@H]1COC(C2=N[C@@H]([C@@H](C)CC)CO2...,Lig2355,in library,NaN,biox,"S,S,S,S"
2049,O=C(C1=CC(C2=NC=CC(C(OCCCCCCCCCC)=O)=C2)=NC=C1...,Lig2356,in library,NaN,bpy,NaN


In [ ]:

#TODO Okay, this is alright since it's very clear it's a series of filtering steps
df = molecular_weight_filter(df)
df = forbidden_substructures_filter(df)
df = multiple_binding_sites_filter(df)
df = missing_bind_site_filter(df)
df = heterocycle_filter(df)
df = isotopes_filter(df)
df = other_metals_filter(df)
display (df)

/var/folders/tw/q0rvjgp93s3_7vdk4652sb540000gn/T/ipykernel_1994/4216922876.py:23: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'molecular weight 582.37 exceeds limit (500.0)' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.at[i, 'status_notes'] = f'molecular weight {weight:.2f} exceeds limit ({weight_limit})'


404 ligands over molecular weight limit were marked for removal.
109 ligands had forbidden substructures and were marked for removal.
0 ligands had multiple binding sites and were marked for removal.
49 ligands are missing a binding site and were marked for removal.
1 ligands contained problematic nitrogen heterocycles and were marked for removal.
76 ligands were duplicates due to isotope encoding and were marked for removal.
4 ligands contained metals and were marked for removal.


,SMILES,id,library_status,status_notes,ligand_class,stereochem
0,C[C@H]1COC(C2=N[C@@H](C)CO2)=N1,Lig1,in library,NaN,biox,"S,S"
1,CC(C)[C@H]1COC(C2=N[C@@H](C(C)C)CO2)=N1,Lig2,in library,NaN,biox,"S,S"
2,CCCC(CCC)[C@H]1COC(C2=N[C@@H](C(CCC)CCC)CO2)=N1,Lig3,in library,NaN,biox,"S,S"
3,CC(C)C[C@H]1COC(C2=N[C@@H](CC(C)C)CO2)=N1,Lig4,in library,NaN,biox,"S,S"
4,c1ccc([C@H]2COC(C3=N[C@@H](c4ccccc4)CO3)=N2)cc1,Lig5,in library,NaN,biox,"S,S"
...,...,...,...,...,...,...
2046,CCCCC(CCCC)[C@H]1COC(C2=N[C@@H](C(CCCC)CCCC)CO...,Lig2353,removed,duplicate when considering isotopes,biox,"S,S"
2047,CC(C)CC(CC(C)C)[C@H]1COC(C2=N[C@@H](C(CC(C)C)C...,Lig2354,removed,duplicate when considering isotopes,biox,"S,S"
2048,CC[C@H](C)[C@H]1COC(C2=N[C@@H]([C@@H](C)CC)CO2...,Lig2355,removed,duplicate when considering isotopes,biox,"S,S,S,S"
2049,O=C(C1=CC(C2=NC=CC(C(OCCCCCCCCCC)=O)=C2)=NC=C1...,Lig2356,removed,molecular weight 524.36 exceeds limit (500.0),bpy,NaN


In [174]:
df.to_excel('/Users/therese/Sigman Group Dropbox/Therese Wild/NN_Library/dft_library_all/temp.xlsx')